<a href="https://colab.research.google.com/github/dacdemon/ITD/blob/main/Copia_de_Coria_Damian_Practica_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Práctica 2: Procesamiento de Reportes de Planta

**Alumno:** Damián Coria

**Comisión:** 2

In [ ]:
# Diagnóstico Inicial:

# Importamos pandas

import pandas as pd

# Cargamos el archivo csv desde un repositorio web

url = "https://raw.githubusercontent.com/dacdemon/ITD/refs/heads/main/practica_2.csv"
df = pd.read_csv(url)


In [ ]:
# buscamos sus dimensiones totales y los tipos de datos
dimensiones = df.shape
tipos_de_datos = df.dtypes

print("Dimensiones del dataset:", dimensiones)
print("tipos de datos:")
print(tipos_de_datos)

Dimensiones del dataset: (4100, 9)
tipos de datos:
ID               int64
Datetime        object
Temperature    float64
Humidity       float64
Pressure       float64
Co2 Gas          int64
PM2.5          float64
PM10           float64
Daytime         object
dtype: object


In [ ]:
# Tratamiento de valores duplicados
duplicados = df.duplicated().sum()
df.drop_duplicates(inplace=True)
duplicados_2 = df.duplicated().sum()
print("valores duplicados encontrados antes de la limpieza: ", duplicados)
print("Valores duplicados encontrados despues de la limpieza: ", duplicados_2)

valores duplicados encontrados antes de la limpieza:  0
Valores duplicados encontrados despues de la limpieza:  0


In [ ]:
# Ajuste de fechas
df["Datetime"] = pd.to_datetime(df["Datetime"])
# Verificamos que se halla realizado el cambio
tiempo = df["Datetime"].dtype
print("El formato de la columna Datetime es: ", tiempo)



El formato de la columna Datetime es:  datetime64[ns]


In [ ]:
# Análisis de tiempo de registros:
# A simple vista los datos parecen ordenados cronológicamente pero por su dimensión no podemos estar seguros asique los ordenamos.

df = df.sort_values("Datetime").reset_index(drop=True)

# contestamos la primer pregunta: ¿Cuál es el período de tiempo total en el que hay registros?

fecha_inicio = df["Datetime"].min()
fecha_fin = df["Datetime"].max()

periodo_total = fecha_fin - fecha_inicio
print("PERÍODOS DE REGISTRO")
print("Primer registro: ", fecha_inicio)
print("Último registro: ", fecha_fin)
print("Período total: ", periodo_total)


PERÍODOS DE REGISTRO
Primer registro:  2019-05-20 19:08:34
Último registro:  2019-05-21 07:02:00
Período total:  0 days 11:53:26


In [ ]:
# Calculamos la frecuencia de muestreo

df["Diferencia"] = df["Datetime"].diff()
print(df["Diferencia"].value_counts().head(10))

print("Promedios de frecuencia:")
print("Frecuencia Promedio :")
print(df["Diferencia"].mean())
print("Frecuencia mediana :")
print(df["Diferencia"].median())

Diferencia
0 days 00:00:02    2012
0 days 00:00:01    1635
0 days 00:00:00     338
0 days 00:00:11      65
0 days 00:00:10      34
0 days 00:00:03       6
0 days 00:00:04       4
0 days 00:12:25       1
0 days 09:43:22       1
0 days 00:04:42       1
Name: count, dtype: int64
Promedios de frecuencia:
Frecuencia Promedio :
0 days 00:00:10.443034886
Frecuencia mediana :
0 days 00:00:02


# ¿ Hay Gaps en los registros ?

La mediana calculada anteriormente representa mejor el intervalo de los datos.
Podemos considerar como Gap cualquier intervalo superior a 2 segundos.

In [ ]:
# Buscaremos si existen diferencias mayores a 2 segundos:

def detectar_gaps(df):
    lista_gaps = []

    for diferencia in df["Diferencia"]:
        if diferencia > pd.Timedelta(seconds=2):
            lista_gaps.append(diferencia)
        else:
          pass

    return lista_gaps
lista_gaps = detectar_gaps(df)

conteo_gaps = len(lista_gaps)
print("Cantidad de gaps encontrados: ", conteo_gaps)


Cantidad de gaps encontrados:  114



# CONSIGNA FINAL DE LA PRACTICA 2: "SI VEN ALGO MÁS, QUE LES RESULTE INTERESANTE EN EL DATASET, SIÉNTANSE LIBRES DE COMPARTIRLO!"
Se Investiga la aparición del primer GAP ya que es una señal de alerta en la recolección de datos.
Si bien un GAP no es la confirmación de un accidente o una falla del equipo, sí representa una interrupción en el registro que debería ser investigada, ya que podría estar relacionada con problemas de comunicación, sensores, etc.



In [ ]:
def detectar_gaps(df):
    lista_gaps = []
    i = 0

    while i < len(df):

        if df.loc[i, "Diferencia"] > pd.Timedelta(seconds=2):
            lista_gaps.append([df.loc[i, "Datetime"], df.loc[i, "Diferencia"]])

        i = i + 1

    return lista_gaps
lista_gaps = detectar_gaps(df)

if lista_gaps:
    print("¡Alerta!")
    print("Primer GAP:",lista_gaps[0][0] )
    print("Duración:",lista_gaps[0][1])
else:
    print("No se encontraron gaps.")

¡Alerta!
Primer GAP: 2019-05-20 19:32:00
Duración: 0 days 00:12:25


## Variables sensadas
Temperature: Corresponde a la medición de la temperatura ambiental en °C (grados Celsius).

Humidity: Corresponde a la medición de la humedad en % (porcentaje de humedad relativa).

Pressure: Corresponde a la medición de la presión en psi .

Co2 Gas: Mide la concentración de dióxido de carbono en ppm (partes por millón).

PM2.5: Mide partículas finas en suspensión en el aire con un diámetro de hasta 2,5 µm.

PM10: Mide partículas finas en suspensión en el aire con un diámetro de hasta 10 µm.

Datetime: Permite conocer el momento exacto en que se realizó cada medición.

ID: Identifica de forma única cada registro.

## Conclusión

A partir del análisis realizado se pudo estudiar el comportamiento de los datos registrados por los sensores.

En primer lugar, se realizó un diagnóstico del dataset mediante ".shape" y ".dtypes". Luego se verificaron y eliminaron los registros duplicados y se convirtió la columna "Datetime" desde texto al formato "datetime64", lo que nos permitió analizar la temporalidad de los datos.

El período analizado es de aproximadamente 11 horas y 53 minutos. El análisis de las diferencias entre registros permitió observar que la frecuencia de muestreo no es constante durante todo el dataset. las frecuencias más comunes son de 1 y 2 segundos, mientras que otras alcanzan o superan los 10 segundos.

La calidad de los datos no son muy buenos porque contienen gran cantidad de gaps y nos indican muchos períodos sin registrar.

Finalmente se destaca la importancia de inspeccionar y limpiar los datos antes de utilizarlos para obtener resultados confiables.